<a href="https://colab.research.google.com/github/hectorjimenez12/CC5205_Proyecto/blob/main/Hito_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [63]:
import pandas as pd
import requests
import seaborn as sns
import datetime

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
pd.set_option('display.max_columns', 50)


# **1.   Analisis de features para predecir exito de un juego**

En esta parte se analiza cuales son los atributos a utlizar en los juegos de steam para generar modelos de prediccion de exito.


In [ ]:
#Dataset de atributos de juegos
url = 'https://raw.githubusercontent.com/hectorjimenez12/Mineria_Datos/main/steam.csv'
df_general = pd.read_csv(url)
df_general

In [ ]:
#Dataset con descripciones de juegos
#!wget https://raw.githubusercontent.com/hectorjimenez12/Mineria_Datos/main/steam_description_data.csv.zip
#!unzip steam_description_data.csv.zip
url = 'https://raw.githubusercontent.com/hectorjimenez12/Mineria_Datos/main/steam_description_data.csv.zip'
df_descriptions = pd.read_csv(url)
df_descriptions

Limpieza de datos de texto

In [158]:
!pip install -qq "spacy >= 3.0.6"

In [159]:
import re
import string
from bs4 import BeautifulSoup
import spacy

nlp = spacy.load("en_core_web_sm") #model for lemmatization

def remove_punctuation(text):
  return ''.join([c for c in text if c not in string.punctuation])

#convierte letras a minusculas si toda
def to_lowercase_if_not_COMPLETEUPPER(text):
  return ' '.join([ i if i.isupper() else i.lower() for i in text.split() ])

def remove_jump_lines(text):
  return text.replace('<br/>', " ").replace('<br>', " ")

#Funcion para eliminar los tags de html
def delete_html_parts(text):
  #change jump line for blank spaces
  soup = BeautifulSoup(text, 'html.parser').get_text()
  return soup

def remove_duplicate_spaces(text):
  return re.sub(' +', ' ', text)

def lemmatization(text):
  doc_nlp = nlp(text)
  return " ".join([token.lemma_ for token in doc_nlp])

def remove_stops(text):
  non_stopwords = []
  doc_nlp = nlp(text)
  for tk in doc_nlp:
    if not doc_nlp.is_stop:
      non_stopwords.append(tk)
  return " ".join(non_stopwords)

remove_stops(df_descriptions['detailed_description'][300])

def clean_text(text):
  text = remove_jump_lines(text)
  text = delete_html_parts(text)
  text = to_lowercase_if_not_COMPLETEUPPER(text)
  text = remove_punctuation(text)
  text = lemmatization(text)
  text = remove_duplicate_spaces(text)
  text = remove_stops(text)
  return text.strip()

print(df_descriptions['detailed_description'][300])
print(clean_text(df_descriptions['detailed_description'][300]))
#soup = BeautifulSoup(df_descriptions['detailed_description'][27331], "html.parser")


AttributeError: ignored

In [ ]:
df_descriptions = df_descriptions[df_descriptions.steam_appid.isin( list(df_general.appid) )]
df_descriptions

In [147]:
df_general['description'] = [clean_text(t)  for t in df_descriptions.detailed_description  ]

KeyboardInterrupt: ignored

# **2.   Implementacion de algoritmos para predecir el exito de los juegos**



Clasificadores para experimentar: (Example sacado de labs)

In [ ]:
from sklearn.datasets import (dataset)
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB  # naive bayes
from sklearn.neighbors import KNeighborsClassifier #kNN
from sklearn.svm import SVC  # support vector machine


c0 = ("Base Dummy", DummyClassifier(strategy='stratified'))
c1 = ("Decision Tree", DecisionTreeClassifier(max_depth=5))
c2 = ("Gaussian Naive Bayes", GaussianNB())
c3 = ("KNN", KNeighborsClassifier(n_neighbors=10))
c4 = ("Support Vector Machines", SVC())

classifiers = [c0, c1, c2, c3, c4]


GridSearch para Decision Tree: (Example)

In [ ]:
tuned_parameters = {'criterion': ['gini', 'entropy'],
                    'max_depth': [3,5,7,10]}

#set scoring metric
score = 'f1'

#Construir el clf con GridSearch
clf = GridSearchCV(DecisionTreeClassifier(criterion=['gini','entropy'], max_depth=[3,5,7,10]),
                   param_grid=tuned_parameters,
                   scoring=score,
                   cv=10)

#Entrenar clf
clf.fit(X_train, y_train)


print("Mejor combinación de parámetros:")
print(clf.best_params_)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))